# Vajra — 01: NWP Data Exploration & Zarr Ingestion
### MoES / NCMRWF Problem Statement 26078
This notebook demonstrates loading 4D numerical weather prediction outputs (time, ensemble member, lat, lon) using Xarray, Dask, and chunked Zarr.

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from vajra.ingestion.nwp_loader import NWPDataLoader
from vajra.ingestion.zarr_writer import ZarrWriter

# Create synthetic 4D ensemble test cube for Bay of Bengal / Indian Subcontinent
ds = NWPDataLoader.create_synthetic_ensemble(
    num_times=8,
    num_members=10,
    lat_range=(5.0, 30.0),
    lon_range=(65.0, 95.0),
    res=0.5
)
print(ds)

In [ ]:
# Export to chunked Zarr store
writer = ZarrWriter(chunks={"time": 4, "number": 5, "latitude": 50, "longitude": 50})
zarr_path = "data/processed/exploration_sample.zarr"
writer.write(ds, zarr_path)
print(f"Exported Zarr cube to: {zarr_path}")

In [ ]:
# Inspect ensemble mean vs spread for Mean Sea Level Pressure (MSLP)
mslp_mean = ds["mslp"].mean(dim="number").isel(time=0)
mslp_std = ds["mslp"].std(dim="number").isel(time=0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
mslp_mean.plot(ax=ax1, cmap="viridis")
ax1.set_title("Ensemble Mean MSLP (Pa)")

mslp_std.plot(ax=ax2, cmap="magma")
ax2.set_title("Ensemble Spread / Uncertainty (Pa)")
plt.tight_layout()
plt.show()